# Prompt evaluation

This notebook runs an end-to-end prompt-evaluation pipeline in two stages.

## Stage 1: dataset generation

`generate_dataset()` asks Claude to produce a small set of AWS-themed tasks.
Each task is a JSON object with three fields:

- `task`: a short natural-language description of the problem.
- `format`: one of `"json"`, `"python"`, or `"regex"` — the shape of the expected solution.
- `solution_criteria`: a plain-English rubric that the model-based grader will use in Stage 2.

The result is saved to `dataset.json`.

## Stage 2: evaluation

`run_eval(dataset)` iterates every test case.
For each one, `run_test_case` does three things:

1. **Solves the task**: `run_prompt(test_case)` asks Claude to produce a candidate solution.
2. **Grades the solution** with two graders:
    - **`grade_by_model`** asks Claude to score the solution against the `solution_criteria`, returning a JSON with `strengths`, `weaknesses`, `reasoning`, and `score` (1-10).
      The prompt frames `<task>`, `<solution>`, and `<criteria>` in XML tags, which Claude attends to very reliably.
    - **`grade_by_syntax`** is a pure-Python validator that scores 10 if the solution parses as its declared `format` (via `json.loads` / `ast.parse` / `re.compile`), 0 otherwise.
      Catches obvious failures that a soft rubric might overlook.
3. Returns a per-case dict with `output`, `test_case`, `score` (the average of the two graders), and `reasoning`.

`run_eval` prints the mean score across the dataset at the end.

## Model split

The pipeline makes three kinds of Claude call, each with a model picked for its role:

- **Dataset generator** (`generate_dataset`): `claude-haiku-4-5` (the default in `models.py`).
  One-shot per eval run, output is inspectable by eye.
- **Solver** (`run_prompt`): `claude-haiku-4-5` (default).
  The model under test; evaluating haiku is the whole point.
- **Grader** (`grade_by_model`): `claude-sonnet-4-5` (explicit override via `model=MODEL_SONNET`).
  Grader quality is the single multiplier in the pipeline: an error here contaminates every evaluation downstream, so the stronger model earns its 5x cost on this one role.


In [1]:
import json

from evals import generate_dataset, run_eval

dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [2]:
with open("dataset.json") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average score: 8.5
[
  {
    "output": "import json\nimport re\n\ndef parse_arn(arn):\n    parts = arn.split(':', 5)\n    if len(parts) != 6 or parts[0] != 'arn':\n        return None\n    \n    return {\n        'partition': parts[1],\n        'service': parts[2],\n        'region': parts[3],\n        'account_id': parts[4],\n        'resource': parts[5]\n    }",
    "test_case": {
      "task": "Write a function that parses an AWS ARN (Amazon Resource Name) and returns a dictionary with its components: partition, service, region, account-id, and resource. For example, 'arn:aws:s3:us-east-1:123456789012:bucket/my-bucket' should return {'partition': 'aws', 'service': 's3', 'region': 'us-east-1', 'account_id': '123456789012', 'resource': 'bucket/my-bucket'}",
      "format": "python",
      "solution_criteria": "Function correctly splits ARN string by colons, handles ARNs with and without region/account-id (e.g., S3 ARNs), returns dictionary with correct keys, handles resource paths tha